# Block 2 Exercise — Generate a Minimal Define-XML v2.1

**Goal:** build a complete, schema-valid Define-XML v2.1 document describing one Demographics
(DM) dataset, entirely from Python objects, and write it to disk.

**Time budget:** ~20 minutes for TODOs 1–4. The Stretch section is optional.

The working cells give you the document skeleton (root, study, metadata version, standards).
The TODOs add the dataset, variables, and codelist, then assemble and write the file. Follow
the worked example in `lecture_notes.md` — it builds exactly this document.

In [1]:
import os
import odmlib.define_2_1.model as DEF

os.makedirs("output", exist_ok=True)

The ODM root element. `Context="Submission"` is the `def:Context` attribute — required in
Define-XML v2.1 (odmlib knows it belongs to the `def:` namespace; you just pass `Context`).
Try removing a required attribute like `FileOID` and re-running: construction fails
immediately with a message naming the missing attribute — errors surface at the source, not
three steps later in a broken file.

In [3]:
odm = DEF.ODM(
    FileOID="DEF.RPH2026.DM",
    FileType="Snapshot",
    CreationDateTime="2026-08-19T12:00:00",
    ODMVersion="1.3.2",
    Context="Submission",
    Originator="R/Pharma 2026 Workshop",
    SourceSystem="odmlib",
)

study = DEF.Study(OID="ST.RPH2026")
study.GlobalVariables = DEF.GlobalVariables(
    StudyName=DEF.StudyName(_content="RPH2026"),
    StudyDescription=DEF.StudyDescription(_content="R/Pharma 2026 odmlib workshop study"),
    ProtocolName=DEF.ProtocolName(_content="RPH-2026-001"),
)

print(study.GlobalVariables.StudyName)

RPH2026


The `MetaDataVersion` (with the required `def:DefineVersion`) and the `def:Standards` section
that every v2.1 define.xml must declare:

In [4]:
mdv = DEF.MetaDataVersion(
    OID="MDV.RPH2026.1",
    Name="RPH2026 Data Definitions",
    Description="Demographics metadata for the workshop",
    DefineVersion="2.1.0",
)

standards = DEF.Standards()
standards.Standard.append(
    DEF.Standard(OID="STD.1", Name="SDTMIG", Type="IG", Version="3.4", Status="Final"))
mdv.Standards = standards

print("MetaDataVersion ready:", mdv.OID)

MetaDataVersion ready: MDV.RPH2026.1


## TODO 1 — The DM dataset and its variable references

Create the `ItemGroupDef` for DM with these attributes:
`OID="IG.DM"`, `Name="DM"`, `Repeating="No"`, `IsReferenceData="No"`, `SASDatasetName="DM"`,
`Domain="DM"`, `Purpose="Tabulation"`, `Structure="One record per subject"`,
`ArchiveLocationID="LF.DM"`, `StandardOID="STD.1"`.

Then give it a `Description` (text "Demographics") and append four `ItemRef`s:

| ItemOID | Mandatory | OrderNumber | KeySequence |
|---------|-----------|-------------|-------------|
| IT.DM.STUDYID | Yes | 1 | 1 |
| IT.DM.USUBJID | Yes | 2 | 2 |
| IT.DM.AGE | No | 3 | — |
| IT.DM.SEX | Yes | 4 | — |

*Hint (lecture §2):* build children in schema order — `Description` first, then the
`ItemRef`s. `Class` and `leaf` come in TODO 2.

In [ ]:
igd = None   # YOUR CODE HERE: create the ItemGroupDef with the attributes above

# YOUR CODE HERE: add the Description and append the four ItemRefs

if igd is None:
    print("TODO 1: create the ItemGroupDef above")
else:
    print(f"{igd.Name}: {len(igd.ItemRef)} ItemRefs")

## TODO 2 — Class and leaf

Every v2.1 dataset declares its class and points at the dataset file with a `def:leaf`:

1. Assign `igd.Class` a `DEF.Class` with `Name="SPECIAL PURPOSE"`.
2. Assign `igd.leaf` a `DEF.leaf` with `ID="LF.DM"`, `href="dm.xpt"`, and a
   `title=DEF.title(_content="dm.xpt")`.

The leaf `ID` must match the `ArchiveLocationID` you set in TODO 1 — that pairing is a
cross-reference the Block 3 checks will verify.

In [ ]:
if igd is None:
    print("Complete TODO 1 first")
else:
    # YOUR CODE HERE: assign igd.Class and igd.leaf
    print("Class:", igd.Class.Name if igd.Class else "TODO",
          "| leaf:", igd.leaf.ID if igd.leaf else "TODO")

## TODO 3 — Variables and the SEX codelist

The `make_item` helper below builds an `ItemDef` with its Description and Origin (in schema
order). Use it to create the four variables, then build the `CL.SEX` codelist:

| OID | Name | DataType | Length | Description | codelist |
|-----|------|----------|--------|-------------|----------|
| IT.DM.STUDYID | STUDYID | text | 12 | Study Identifier | — |
| IT.DM.USUBJID | USUBJID | text | 25 | Unique Subject Identifier | — |
| IT.DM.AGE | AGE | integer | 3 | Age | — |
| IT.DM.SEX | SEX | text | 1 | Sex | CL.SEX |

For the codelist: `DEF.CodeList(OID="CL.SEX", Name="Sex", DataType="text")` with two
`CodeListItem`s — `F` decoding to `Female`, `M` to `Male`.

*Hint (lecture §2):* a term is a `CodeListItem(CodedValue=...)` whose `Decode` holds a
`TranslatedText`. Append items to `mdv.ItemDef` and the codelist to `mdv.CodeList`.

In [ ]:
def make_item(oid, name, dtype, length, desc, codelist=None):
    item = DEF.ItemDef(OID=oid, Name=name, DataType=dtype, Length=length, SASFieldName=name)
    item.Description = DEF.Description()
    item.Description.TranslatedText.append(DEF.TranslatedText(_content=desc, lang="en"))
    if codelist:
        item.CodeListRef = DEF.CodeListRef(CodeListOID=codelist)
    item.Origin.append(DEF.Origin(Type="Collected"))
    return item

# YOUR CODE HERE: create the four ItemDefs with make_item and append them to mdv.ItemDef

# YOUR CODE HERE: build the CL.SEX CodeList and append it to mdv.CodeList

print(f"{len(mdv.ItemDef)} ItemDefs, {len(mdv.CodeList)} CodeLists")

## TODO 4 — Assemble and write

Wire everything together and write the file:

1. Append `igd` to `mdv.ItemGroupDef`.
2. Assign `study.MetaDataVersion = mdv` — **assignment, not append**: in Define-XML, `Study`
   and `MetaDataVersion` are single objects.
3. Assign `odm.Study = study`.
4. `odm.write_xml("output/define_dm.xml")`

In [ ]:
if igd is None or not mdv.ItemDef:
    print("Complete TODOs 1-3 first")
else:
    # YOUR CODE HERE: assemble the document and write output/define_dm.xml
    pass

A first taste of Block 3: the OID checker verifies every cross-reference in the document you
just built — `ItemRef → ItemDef`, `CodeListRef → CodeList`, `ArchiveLocationID → leaf`:

In [ ]:
from odmlib import create_oid_checker

if getattr(odm, "Study", None) is None:
    print("Complete TODO 4 first")
else:
    odm.verify_oids(create_oid_checker("define_2_1"))
    print("All OID references resolve - your define.xml is internally consistent")

Open `output/define_dm.xml` in the JupyterLab editor and skim it: four namespaces, schema
ordering, prefixed attributes — all handled for you.

## Stretch — value-level metadata

Value-level metadata says "variable X has different definitions depending on a condition" —
for example, AGE recorded in years vs months. The machinery is a `ValueListDef` (the
value-level items) plus `WhereClauseDef`s (the conditions), linked with `WhereClauseRef` and
`ValueListRef`.

This code adds an AGEU variable, then value-level metadata for AGE when AGEU = YEARS. Note
what happens at the end: because we're adding `ValueListDef`/`WhereClauseDef` to an
already-built document, they land out of schema order — `verify_order()` flags it and
`reorder_object()` on the flagged element fixes it.

In [ ]:
from odmlib.exceptions import OdmlibElementOrderError

if getattr(odm, "Study", None) is None:
    print("Complete TODO 4 first")
else:
    # a normal AGEU variable
    mdv.ItemDef.append(make_item("IT.DM.AGEU", "AGEU", "text", 6, "Age Units"))
    igd.ItemRef.append(DEF.ItemRef(ItemOID="IT.DM.AGEU", Mandatory="No", OrderNumber=5))

    # the value-level item: AGE when it is recorded in years
    mdv.ItemDef.append(make_item("IT.DM.AGE.YEARS", "AGE", "integer", 3, "Age in Years"))

    # the condition: AGEU = "YEARS"
    wcd = DEF.WhereClauseDef(OID="WC.DM.AGEU.YEARS")
    check = DEF.RangeCheck(SoftHard="Soft", ItemOID="IT.DM.AGEU", Comparator="EQ")
    check.CheckValue.append(DEF.CheckValue(_content="YEARS"))
    wcd.RangeCheck.append(check)
    mdv.WhereClauseDef.append(wcd)

    # the value list: one value-level ItemRef guarded by the condition
    vld = DEF.ValueListDef(OID="VL.DM.AGE")
    vref = DEF.ItemRef(ItemOID="IT.DM.AGE.YEARS", Mandatory="No", OrderNumber=1)
    vref.WhereClauseRef.append(DEF.WhereClauseRef(WhereClauseOID="WC.DM.AGEU.YEARS"))
    vld.ItemRef.append(vref)
    mdv.ValueListDef.append(vld)

    # point AGE at its value-level metadata
    mdv.find("ItemDef", "OID", "IT.DM.AGE").ValueListRef = DEF.ValueListRef(ValueListOID="VL.DM.AGE")

    # late additions land out of schema order - verify, then repair
    try:
        odm.verify_order()
    except OdmlibElementOrderError as oe:
        print("order issue:", oe)
        mdv.reorder_object()
        odm.verify_order()
        print("fixed with reorder_object()")

    odm.verify_oids(create_oid_checker("define_2_1"))
    odm.write_xml("output/define_dm_vlm.xml")
    print("wrote output/define_dm_vlm.xml")

**Done?** Compare with `solutions/create_define_solution.ipynb`. In Block 3, the file you just
wrote goes through all four validation layers.